In [ ]:
import requests
import pandas as pd
import json
from py_mini_racer import py_mini_racer

URL = "https://play.pokemonshowdown.com/data/pokedex.js"

# Descargar el JS
js_code = requests.get(URL).text

# Ejecutar JS en V8
ctx = py_mini_racer.MiniRacer()
ctx.eval("var exports = {};")
ctx.eval(js_code)

json_str = ctx.eval("JSON.stringify(exports.BattlePokedex)")

if json_str is None:
    raise RuntimeError("No se pudo cargar BattlePokedex")

pokedex = json.loads(json_str)

# Convertir a dataFrame
rows = []

for key, p in pokedex.items():
    stats = p.get("baseStats", {})
    abilities = p.get("abilities", {})

    rows.append({
        "internal_id": key,
        "pokedex_number": p.get("num"),
        "name": p.get("name"),
        "types": ",".join(p.get("types", [])),
        "hp": stats.get("hp"),
        "attack": stats.get("atk"),
        "defense": stats.get("def"),
        "sp_attack": stats.get("spa"),
        "sp_defense": stats.get("spd"),
        "speed": stats.get("spe"),
        "abilities": ",".join(abilities.values()),
        "height_m": p.get("heightm"),
        "weight_kg": p.get("weightkg"),
        "color": p.get("color"),
        "egg_groups": ",".join(p.get("eggGroups", [])),
        "generation": p.get("gen"),
        "tier": p.get("tier"),
        "base_species": p.get("baseSpecies"),
        "forme": p.get("forme"),
        "is_mega": p.get("isMega", False),
        "is_gmax": p.get("isGigantamax", False)
    })

df = pd.DataFrame(rows)


# Guardar la base de datos

df = df.sort_values(["pokedex_number", "name"])
df.reset_index(drop=True, inplace=True)

# Guardar el DataFrame en un archivo CSV
ruta_salida = "../data/pokedex_showdown.csv"
df.to_csv(ruta_salida, index=False, encoding='utf-8')


